# Fruits Vector Demo — Student Lab

In this lab you will:
1. Create fruit vectors manually (based on color: red, yellow, green)
2. Store them in LanceDB
3. Compare different similarity metrics
4. Visualize the vectors in 3D space

**Instructions**: Fill the `<replace_here>` placeholders. Check `guide.md` for solutions.

## Cell 1: Imports

We need these libraries:
- **numpy**: For vector math
- **lancedb**: Vector database
- **pyarrow**: For defining schema
- **pathlib**: For file paths

In [ ]:
import numpy as np
import lancedb
import pyarrow as pa
from pathlib import Path

DB_PATH = Path("fruits_lancedb")
TABLE_NAME = "fruit_vectors"
DB_PATH.mkdir(exist_ok=True)
print(f"✅ Database ready: {DB_PATH}")

## Cell 2: Create Vector Table

We need to define a schema for our table:
- `name`: Fruit name (string)
- `vector`: 3D color vector (list of 3 floats)

**Your task**: Fill in the PyArrow types.

In [ ]:
db = lancedb.connect(str(DB_PATH))

if TABLE_NAME in db.table_names():
    db.drop_table(TABLE_NAME)

schema = pa.schema([
    pa.field("name", pa.<replace_here>),
    pa.field("vector", pa.list_(pa.<replace_here>, 3))
])

table = db.create_table(TABLE_NAME, schema=schema)
print(f"✅ Table created: {TABLE_NAME}")

## Cell 3: Define and Store Fruit Vectors

We manually create 3D vectors for 8 fruits. Each vector represents color intensity:
- Dimension 1: Red intensity
- Dimension 2: Yellow intensity  
- Dimension 3: Green intensity

**Your task**: Fill in the method to add records.

In [ ]:
fruit_vectors = {
    "papaya": np.array([0.88, 0.35, 0.05]),
    "peach": np.array([0.81, 0.32, 0.12]),
    "pear": np.array([0.62, 0.51, 0.38]),
    "pineapple": np.array([0.15, 0.85, 0.79]),
    "apple": np.array([0.28, 0.42, 0.69]),
    "apricot": np.array([0.77, 0.44, 0.2]),
    "guava": np.array([0.1, 0.74, 0.58]),
    "nectarine": np.array([0.83, 0.3, 0.1])
}

records = [{"name": name, "vector": vector.tolist()} for name, vector in fruit_vectors.items()]
table.<replace_here>(records)
print(f"✅ Inserted {len(records)} fruits")

## Cell 4: Verify Stored Data

Let's check what we stored in the database.

In [ ]:
stored = table.to_pandas()
print("📦 Stored fruits:", stored["name"].tolist())

## Cell 5: Define Mango Query Vector

We'll use this vector to find the fruit most similar to mango.
Mango has: high red, medium yellow, very low green.

In [ ]:
mango_query = np.array([0.9, 0.34, 0.08])
print(f"🍋 Mango query: {mango_query}")

## Cell 6: Compare Similarity Metrics

We compare 3 different ways to measure similarity:
- **Cosine similarity**: Measures angle between vectors
- **Euclidean distance**: Straight-line distance
- **Dot product**: Projection of one vector onto another

**Your task**: Fill in the numpy function for dot product.

In [ ]:
cosine_sim = lambda a, b: np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
euclidean_dist = lambda a, b: np.linalg.norm(a - b)
dot_prod = lambda a, b: np.<replace_here>(a, b)

metrics = [
    ("Cosine", sorted([(n, cosine_sim(v, mango_query)) for n, v in fruit_vectors.items()], key=lambda x: x[1], reverse=True)),
    ("Euclidean", sorted([(n, euclidean_dist(v, mango_query)) for n, v in fruit_vectors.items()], key=lambda x: x[1])),
    ("Dot", sorted([(n, dot_prod(v, mango_query)) for n, v in fruit_vectors.items()], key=lambda x: x[1], reverse=True))
]

for name, ranking in metrics:
    print(f"\n🔹 {name}: {', '.join([f'{r[0]}({r[1]:.2f})' for r in ranking[:3]])}")

## Cell 7: Visualize in 3D Space

Let's plot the fruits in 3D color space! The red star is our mango query vector.
Notice how similar-colored fruits cluster together.

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

names = list(fruit_vectors.keys())
vecs = np.array(list(fruit_vectors.values()))

fig = plt.figure(figsize=(10, 4))

ax1 = fig.add_subplot(121, projection='3d')
ax1.scatter(vecs[:,0], vecs[:,1], vecs[:,2], c='blue', s=80, alpha=0.6)
ax1.scatter(mango_query[0], mango_query[1], mango_query[2], c='red', s=200, marker='*')
for i, n in enumerate(names): ax1.text(vecs[i,0], vecs[i,1], vecs[i,2], n, fontsize=8)
ax1.set_xlabel('Red'); ax1.set_ylabel('Yellow'); ax1.set_zlabel('Green')
ax1.set_title('3D Color Space')

ax2 = fig.add_subplot(122)
ax2.scatter(vecs[:,0], vecs[:,1], c='blue', s=80, alpha=0.6)
ax2.scatter(mango_query[0], mango_query[1], c='red', s=200, marker='*')
for i, n in enumerate(names): ax2.annotate(n, (vecs[i,0], vecs[i,1]), fontsize=8)
ax2.set_xlabel('Red'); ax2.set_ylabel('Yellow')
ax2.set_title('Red vs Yellow'); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("✅ Lab complete!")

## 🎉 Congratulations!

You've completed the Fruits Vector Demo! 

**What you learned:**
- How to create vectors manually (color-based)
- How to store them in LanceDB
- How 3 different similarity metrics compare
- How to visualize vectors in 3D space

**Key insight**: Papaya is the top match for mango across all metrics because they have similar colors!